# Image-to-Image генерация
## Stable Diffusion (бесплатно, без цензуры)

In [ ]:
# Установка
!pip install diffusers transformers accelerate safetensors pillow requests
!pip install -q git+https://github.com/huggingface/diffusers.git

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline, DiffusionPipeline
from PIL import Image
import requests
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Проверка GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")
if device == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, VRAM: {gpu.total_memory/1024**3:.1f} GB")

In [ ]:
# Выбери модель
# 1. uncensored - без цензуры
# 2. sd-1.5 - обычная (может ценизурить)

MODEL_CHOICE = "uncensored"  # поменяй на "sd15" если нужно

if MODEL_CHOICE == "uncensored":
    # Модель без цензуры (Darkstorm)
    model_id = "darkstorm2150/Protogen_x3.4_Official_Release"
else:
    model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,          # отключаем цензуру
    requires_safety_checker=False
)
pipe = pipe.to(device)
print("Модель загружена")

In [ ]:
def generate(init_img, prompt, strength=0.7, steps=25):
    """Генерирует изображение на основе исходного + промпт"""
    result = pipe(
        prompt=prompt,
        image=init_img,
        strength=strength,
        num_inference_steps=steps,
        guidance_scale=7.5
    ).images[0]
    return result

In [ ]:
# === ЗАГРУЗКА ИЗОБРАЖЕНИЯ ===
# Вариант 1: ссылка на картинку
image_url = ""  # вставь URL сюда

# Вариант 2: загрузить файл (раскомментируй)
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
init_image = Image.open(filename).convert("RGB")
init_image.thumbnail((768, 768))
display(init_image)

In [ ]:
# === ПРОМПТ ===
prompt = ""  # введи описание желаемого результата
""
negative_prompt = ""  # что исключить
strength = 0.7          # 0.1 = почти как оригинал, 0.9 = сильно изменить
steps = 30              # качество (чем больше, тем дольше)

result = generate(init_image, prompt, strength, steps)
display(result)

# Сохранить
result.save("output.png")
print("Сохранено: output.png")

In [ ]:
# Скачать результат
files.download("output.png")

---
**⚠️ Важно:** Colab бесплатно даёт T4 GPU (~15 часов в месяц).
При long ожидании Colab может отключить runtime. Сохраняй результат сразу.
**Не генерируй контент, нарушающий законы РФ.**